In [2]:
import os
from dotenv import load_dotenv
load_dotenv() # aloading all the enviroment variable

groq_api_key = os.getenv("GROQ_API_KEY")


In [3]:
from langchain_groq import ChatGroq

model = ChatGroq(model="groq/compound",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x7fa9f903fc70>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7faa39478250>, model_name='groq/compound', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [4]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi I am rohit and I am a System engineer ")])

AIMessage(content='Hi Rohit! Nice to meet you. How can I assist you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 254, 'total_tokens': 322, 'completion_time': 0.141872, 'completion_tokens_details': None, 'prompt_time': 0.008917, 'prompt_tokens_details': None, 'queue_time': 0.104563, 'total_time': 0.150789}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019e4304-b17e-7ee0-80cb-ef74c38c64e6-0', usage_metadata={'input_tokens': 254, 'output_tokens': 68, 'total_tokens': 322})

In [5]:
from langchain_core.messages import AIMessage

model.invoke(
    [
        HumanMessage(content="Hi I am rohit and I am a System engineer"),
        AIMessage(content="Hi Rohit! Nice to meet you. How can I assist you today?"),
        HumanMessage(content="Who am I? What is my name?")
    ]
)

AIMessage(content='**Reasoning**\n\n1. In your first message you wrote:  \n   *“Hi I am rohit and I am a System engineer.”*  \n   This statement directly gives two pieces of information about you:\n   - Your name is **Rohit**.  \n   - Your professional role is **System Engineer**.\n\n2. When you later asked, “Who am I? What is my name?” the answer can be derived straight from that earlier self‑identification.\n\n**Answer**\n\n- **Name:** Rohit  \n- **Who you are:** A System Engineer (i.e., you work as a system engineer).', additional_kwargs={'reasoning_content': '\nTo answer the question "Who am I? What is my name?", I need to recall the information provided by the user earlier in the conversation.\n\nThe user previously mentioned that their name is Rohit and that they are a System Engineer.\n\n\n\nYour name is Rohit, and you are a System Engineer.'}, response_metadata={'token_usage': {'completion_tokens': 321, 'prompt_tokens': 1061, 'total_tokens': 1382, 'completion_time': 0.694595, '

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keepp track of inputs and outputs of the model, and store them in some datastore.
Future interaction will then load those messages and pass them into the chain as part of the input.

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    for key,val in store.items():
        
        print(key,"->",val)
        
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [7]:
config = {"configurable":{"session_id":"chat1"}}

In [9]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, My name is Rohit and I am a System Engineer")],
    config=config
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?


In [10]:
response.content

'Nice to meet you, Rohit! As a System Engineer, you probably work with complex hardware and software integrations. How can I help you today?'

In [13]:
response = with_message_history.invoke(
    [HumanMessage(content="Who am I?")],
    config=config
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Hi, My name is Rohit and I am a System Engineer
AI: Nice to meet you, Rohit! As a System Engineer, you probably work with complex hardware and software integrations. How can I help you today?
Human: Who am I?
AI: **Answer – Who you are**

Based on the information you’ve shared in this conversation:

1. **Name:** You introduced yourself as **Rohit**.  
2. **Profession:** You stated that you work as a **System Engineer**.

**Reasoning**

- In your first message you wrote, “Hi, My name is Rohit and I am a System Engineer.”  
- I acknowledged that statement in subsequent replies, confirming that you are Rohit and that your role involves working with complex hardware and software integrations.  
- When you asked “Who am I?” I referred back to those two explicit pieces of information you provided.

**Conclusion**

You are **Rohit, a System Engineer**.


In [12]:
response.content

'**Answer –\u202fWho you are**\n\nBased on the information you’ve shared in this conversation:\n\n1. **Name:** You introduced yourself as **Rohit**.  \n2. **Profession:** You stated that you work as a **System Engineer**.\n\n**Reasoning**\n\n- In your first message you wrote, “Hi, My name is Rohit and I am a System Engineer.”  \n- I acknowledged that statement in subsequent replies, confirming that you are Rohit and that your role involves working with complex hardware and software integrations.  \n- When you asked “Who am I?” I referred back to those two explicit pieces of information you provided.\n\n**Conclusion**\n\nYou are **Rohit, a System Engineer**.'

In [14]:
config1 = {"configurable":{"session_id":"chat2"}}
response = with_message_history.invoke(
    [HumanMessage(content="Who am I")],
    config=config1
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Hi, My name is Rohit and I am a System Engineer
AI: Nice to meet you, Rohit! As a System Engineer, you probably work with complex hardware and software integrations. How can I help you today?
Human: Who am I?
AI: **Answer – Who you are**

Based on the information you’ve shared in this conversation:

1. **Name:** You introduced yourself as **Rohit**.  
2. **Profession:** You stated that you work as a **System Engineer**.

**Reasoning**

- In your first message you wrote, “Hi, My name is Rohit and I am a System Engineer.”  
- I acknowledged that statement in subsequent replies, confirming that you are Rohit and that your role involves working with complex hardware and software integrations.  
- When you asked “Who am I?” I referred back to those two explicit pieces of information you provided.

**Conclusion**

You are **Rohit, a System Engineer**.
Human: W

In [15]:
response.content

'**Answer**\n\nI’m not able to determine your personal identity.\n\n**Why**\n\n- The only information I have about you is the text you’ve typed in this conversation.  \n- No identifying details (name, email, user‑account data, etc.) were provided, and I have no access to external systems that could look up a “who‑are‑you” record.  \n- The web‑search results I retrieved discuss the general concepts of *user identity*, *authentication*, and *access management*, but none of them contain any specific data about you as an individual.\n\n**What this means**\n\n- From a technical standpoint, I can only say that you are *the person currently interacting with me via this text‑based interface*.  \n- If you’d like a more personalized answer, you’ll need to share additional context (e.g., a name, role, or any other identifier you’re comfortable providing).  \n\nOtherwise, the most accurate response I can give is that I don’t have enough information to answer “Who am I?” for you specifically.'

In [16]:
response = with_message_history.invoke(
    [HumanMessage(content="My name is Jhon the Don")],
    config=config1
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Hi, My name is Rohit and I am a System Engineer
AI: Nice to meet you, Rohit! As a System Engineer, you probably work with complex hardware and software integrations. How can I help you today?
Human: Who am I?
AI: **Answer – Who you are**

Based on the information you’ve shared in this conversation:

1. **Name:** You introduced yourself as **Rohit**.  
2. **Profession:** You stated that you work as a **System Engineer**.

**Reasoning**

- In your first message you wrote, “Hi, My name is Rohit and I am a System Engineer.”  
- I acknowledged that statement in subsequent replies, confirming that you are Rohit and that your role involves working with complex hardware and software integrations.  
- When you asked “Who am I?” I referred back to those two explicit pieces of information you provided.

**Conclusion**

You are **Rohit, a System Engineer**.
Human: W

In [17]:
response = with_message_history.invoke(
    [HumanMessage(content="Who am I")],
    config=config1
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Hi, My name is Rohit and I am a System Engineer
AI: Nice to meet you, Rohit! As a System Engineer, you probably work with complex hardware and software integrations. How can I help you today?
Human: Who am I?
AI: **Answer – Who you are**

Based on the information you’ve shared in this conversation:

1. **Name:** You introduced yourself as **Rohit**.  
2. **Profession:** You stated that you work as a **System Engineer**.

**Reasoning**

- In your first message you wrote, “Hi, My name is Rohit and I am a System Engineer.”  
- I acknowledged that statement in subsequent replies, confirming that you are Rohit and that your role involves working with complex hardware and software integrations.  
- When you asked “Who am I?” I referred back to those two explicit pieces of information you provided.

**Conclusion**

You are **Rohit, a System Engineer**.
Human: W

In [18]:
response.content

'You’re Jhon\u202fthe\u202fDon. If there’s anything specific you’d like to discuss or ask about, just let me know!'

# Prompt Template
Prompt Template help to turn raw user information into a format that the LLM can work with. In case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [23]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant. Answer the questions to the next of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [25]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Rohit")]})

AIMessage(content='Hi Rohit! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 279, 'total_tokens': 344, 'completion_time': 0.136497, 'completion_tokens_details': None, 'prompt_time': 0.011084, 'prompt_tokens_details': None, 'queue_time': 0.106251, 'total_time': 0.14758}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019e4dec-b5a7-77b1-9116-54153010da88-0', usage_metadata={'input_tokens': 279, 'output_tokens': 65, 'total_tokens': 344})

In [26]:
with_message_history = RunnableWithMessageHistory(chain,get_session_history)

In [29]:
config={"configurable":{"session_id":"chat3"}}
response = with_message_history.invoke(
    [HumanMessage(content="Hi my name is Rohit")],
    config=config
)

chat1 -> Human: Hi, My name is Rohit and I am a System Engineer
AI: Hello Rohit! Nice to meet you. How can I assist you today?
Human: Hi, My name is Rohit and I am a System Engineer
AI: Nice to meet you, Rohit! As a System Engineer, you probably work with complex hardware and software integrations. How can I help you today?
Human: Who am I?
AI: **Answer – Who you are**

Based on the information you’ve shared in this conversation:

1. **Name:** You introduced yourself as **Rohit**.  
2. **Profession:** You stated that you work as a **System Engineer**.

**Reasoning**

- In your first message you wrote, “Hi, My name is Rohit and I am a System Engineer.”  
- I acknowledged that statement in subsequent replies, confirming that you are Rohit and that your role involves working with complex hardware and software integrations.  
- When you asked “Who am I?” I referred back to those two explicit pieces of information you provided.

**Conclusion**

You are **Rohit, a System Engineer**.
Human: W

In [30]:
response

AIMessage(content='Hello Rohit! Nice to meet you. How can I help you today?', additional_kwargs={'reasoning_content': '<Think>\n\n</Think>'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 279, 'total_tokens': 336, 'completion_time': 0.12884, 'completion_tokens_details': None, 'prompt_time': 0.013087, 'prompt_tokens_details': None, 'queue_time': 0.131767, 'total_time': 0.141928}, 'model_name': 'groq/compound', 'system_fingerprint': None, 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None}, id='run--019e4df1-df3f-7520-b853-b05a39d7d108-0', usage_metadata={'input_tokens': 279, 'output_tokens': 57, 'total_tokens': 336})